# Feature Engineering

This notebook focuses on preparing the dataset for machine learning models used in fashion trend prediction.

The goal of feature engineering is to transform raw variables into more informative representations that help models better capture demand patterns and trend dynamics.

Key operations performed in this notebook include:

* Loading the cleaned dataset from the exploratory data analysis stage
* Creating lag-based features that capture historical demand behavior
* Generating rolling statistics to represent short-term momentum
* Encoding categorical variables for machine learning compatibility
* Preparing the final dataset for modeling

These engineered features will allow the prediction models to capture temporal patterns, demand momentum, and structural characteristics of fashion product trends.


In [33]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
df = pd.read_csv("../data/webscrapped_raw_dataset/fashion_trend_weekly_dataset_2021_2025.csv")

print(df.shape)
df.head()

(1434, 26)


,week,year,month,season,category,search_index,search_growth,momentum_4w,keyword_frequency,brand_popularity,...,dark_ratio,bright_ratio,pattern_printed_ratio,pattern_plain_ratio,image_count,availability_count,discount_avg,new_arrival_ratio,trend_label,next_week_search
0,1,2021,1,Winter,Hoodie,51.38,0.000000,52.510,157,0.63,...,0.55,0.00,0.11,0.89,283,293,0.16,0.15,1,54.73
1,2,2021,1,Winter,Hoodie,54.73,0.065200,52.510,157,0.84,...,0.30,0.37,0.31,0.69,281,299,0.21,0.17,2,48.89
2,3,2021,1,Winter,Hoodie,48.89,-0.106706,52.510,151,0.77,...,0.52,0.04,0.20,0.80,232,262,0.17,0.09,0,55.04
3,4,2021,1,Winter,Hoodie,55.04,0.125793,52.510,153,0.54,...,0.44,0.00,0.10,0.90,259,296,0.16,0.16,2,56.72
4,5,2021,2,Winter,Hoodie,56.72,0.030523,53.845,150,0.46,...,0.70,0.00,0.13,0.87,252,329,0.21,0.09,1,62.65


In [35]:
print(df.shape)

(1434, 26)


In [36]:
df = df.sort_values(by=["category", "year", "week"]).reset_index(drop=True)

In [37]:
df = df.sort_values(by=["category", "year", "week"]).reset_index(drop=True)

df["lag_1_search"] = df.groupby("category")["search_index"].shift(1)
df["lag_2_search"] = df.groupby("category")["search_index"].shift(2)
df["lag_3_search"] = df.groupby("category")["search_index"].shift(3)

## Rolling Momentum Features

Rolling features summarize recent historical behavior across a time window.

In fashion trend prediction, short-term demand momentum often signals whether a style is gaining or losing popularity. Rolling statistics help capture these dynamics by computing averages and variation across the past few weeks.

Here we compute rolling statistics over a 4-week window to represent short-term trend momentum.


In [38]:
df["rolling_mean_4w"] = (
    df.groupby("category")["search_index"]
      .rolling(window=4)
      .mean()
      .reset_index(level=0, drop=True)
)

df["rolling_std_4w"] = (
    df.groupby("category")["search_index"]
      .rolling(window=4)
      .std()
      .reset_index(level=0, drop=True)
)

In [39]:
df[[
    "category",
    "week",
    "search_index",
    "rolling_mean_4w",
    "rolling_std_4w"
]].head(12)

,category,week,search_index,rolling_mean_4w,rolling_std_4w
0,Hoodie,1,51.38,NaN,NaN
1,Hoodie,2,54.73,NaN,NaN
2,Hoodie,3,48.89,NaN,NaN
3,Hoodie,4,55.04,52.5100,2.927490
4,Hoodie,5,56.72,53.8450,3.417060
5,Hoodie,6,62.65,55.8250,5.659567
6,Hoodie,7,58.93,58.3350,3.288287
7,Hoodie,8,64.61,60.7275,3.561876
8,Hoodie,9,63.18,62.3425,2.420928
9,Hoodie,10,67.53,63.5625,3.579789


## Handling Missing Values from Feature Engineering

Lag features and rolling statistics rely on historical observations.
Because the first few rows of each category do not have sufficient past data, these rows contain missing values.

To ensure that machine learning models receive complete feature vectors, rows containing missing values generated during feature engineering are removed.


In [40]:
df = df.dropna().reset_index(drop=True)

print(df.shape)

(1416, 31)


## Encoding Categorical Variables

Machine learning algorithms require numerical input features.
The variables `season` and `category` are categorical, so they must be converted into numerical form.

One-hot encoding creates binary indicator columns representing each category level. This allows models to learn the influence of different seasons and clothing categories on fashion trend dynamics.


In [41]:
df = pd.get_dummies(df, columns=["season", "category"], drop_first=True)

print(df.shape)
df.head()

(1416, 37)


,week,year,month,search_index,search_growth,momentum_4w,keyword_frequency,brand_popularity,avg_price,avg_rating,...,rolling_mean_4w,rolling_std_4w,season_Monsoon,season_Summer,season_Winter,category_Jeans,category_Kurti,category_Saree,category_Streetwear,category_TShirt
0,4,2021,1,55.04,0.125793,52.5100,153,0.54,1970.19,3.29,...,52.5100,2.927490,False,False,True,False,False,False,False,False
1,5,2021,2,56.72,0.030523,53.8450,150,0.46,1758.52,4.40,...,53.8450,3.417060,False,False,True,False,False,False,False,False
2,6,2021,2,62.65,0.104549,55.8250,189,0.89,2190.03,4.72,...,55.8250,5.659567,False,False,True,False,False,False,False,False
3,7,2021,2,58.93,-0.059377,58.3350,186,0.46,1803.50,4.03,...,58.3350,3.288287,False,False,True,False,False,False,False,False
4,8,2021,2,64.61,0.096386,60.7275,201,0.84,1884.70,3.32,...,60.7275,3.561876,False,False,True,False,False,False,False,False


In [42]:
X = df.drop(columns=["trend_label", "next_week_search"])
y_class = df["trend_label"]
y_reg = df["next_week_search"]

print(X.shape)

(1416, 35)


In [43]:
bool_cols = X.select_dtypes(include="bool").columns

X[bool_cols] = X[bool_cols].astype(int)

In [44]:
engineered_df = pd.concat([X, y_class, y_reg], axis=1)

print(engineered_df.shape)

(1416, 37)


In [45]:
engineered_df.to_csv(
    "../data/processed_data/fashion_trend_engineered.csv",
    index=False
)

In [46]:
import os
os.listdir("../data/processed_data")

['fashion_trend_engineered.csv', 'nlp_data', 'visual_data']